In [1]:
import pandas as pd
import numpy as np

# 0 - Dataset

Source: https://www.kaggle.com/datasets/devrishisharma/finguard-ai-behavioral-fraud-detection-dataset

### Nhắc lại cấu trúc dữ liệu

| Bảng | Khóa chính | Khóa ngoại | Mô tả |
|:---|:---|:---|:---|
| `users` | `user_id` | — | Thông tin khách hàng |
| `accounts` | `account_id` | `user_id` | Tài khoản ngân hàng |
| `transactions` | `txn_id` | `account_id` | Giao dịch |

```mermaid
erDiagram
    USERS ||--o{ ACCOUNTS : "has"
    ACCOUNTS ||--o{ TRANSACTIONS : "has"
```

### Mối quan hệ:
- Một **user** có thể có nhiều **account**
- Một **account** có thể có nhiều **transaction**

In [2]:
# Download dataset từ Kaggle
import kagglehub
import os

path = kagglehub.dataset_download("devrishisharma/finguard-ai-behavioral-fraud-detection-dataset")
print("Path to dataset files:", path)

/Users/dz/dev/Coding101/Coding-101/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/dz/.cache/kagglehub/datasets/devrishisharma/finguard-ai-behavioral-fraud-detection-dataset/versions/1


In [3]:
# Load dữ liệu
users = pd.read_csv(os.path.join(path, 'users.csv'))
accounts = pd.read_csv(os.path.join(path, 'accounts.csv'))
transactions = pd.read_csv(os.path.join(path, 'transactions.csv'))

print(f"users: {users.shape[0]} dòng, {users.shape[1]} cột")
print(f"accounts: {accounts.shape[0]} dòng, {accounts.shape[1]} cột")
print(f"transactions: {transactions.shape[0]} dòng, {transactions.shape[1]} cột")

users: 5000 dòng, 8 cột
accounts: 7468 dòng, 6 cột
transactions: 221514 dòng, 10 cột


## Dữ liệu minh họa (Demo)

Để dễ hình dung các khái niệm merge và pivot, ta tạo thêm một bộ dữ liệu nhỏ mô phỏng **hệ thống trường học**.

| Bảng | Mô tả |
|:---|:---|
| `df_students` | Danh sách học sinh |
| `df_classes` | Danh sách lớp học |
| `df_scores` | Bảng điểm thi |

### Lưu ý thiết kế:
- **S5** thuộc lớp **C3** → nhưng C3 **không có** trong bảng lớp
- Lớp **C4** tồn tại → nhưng **không có** học sinh nào
- **S6** có điểm → nhưng **không có** trong bảng học sinh

→ Các trường hợp "lệch" này giúp minh họa rõ sự khác biệt giữa các kiểu merge.

In [4]:
# Bảng học sinh
df_students = pd.DataFrame({
    'student_id': ['S1', 'S2', 'S3', 'S4', 'S5'],
    'name': ['An', 'Bình', 'Chi', 'Dũng', 'Em'],
    'class_id': ['C1', 'C1', 'C2', 'C2', 'C3']
})

# Bảng lớp học
df_classes = pd.DataFrame({
    'class_id': ['C1', 'C2', 'C4'],
    'class_name': ['Toán nâng cao', 'Lý cơ bản', 'Hóa nâng cao'],
    'teacher': ['Thầy Hùng', 'Cô Mai', 'Thầy Tú']
})

# Bảng điểm thi
df_scores = pd.DataFrame({
    'student_id': ['S1', 'S1', 'S2', 'S3', 'S3', 'S6'],
    'subject': ['Toán', 'Lý', 'Toán', 'Toán', 'Lý', 'Toán'],
    'score': [9, 8, 7, 6, 9, 10]
})

print("=== df_students ===")
display(df_students)
print("\n=== df_classes ===")
display(df_classes)
print("\n=== df_scores ===")
display(df_scores)

=== df_students ===


,student_id,name,class_id
0,S1,An,C1
1,S2,Bình,C1
2,S3,Chi,C2
3,S4,Dũng,C2
4,S5,Em,C3



=== df_classes ===


,class_id,class_name,teacher
0,C1,Toán nâng cao,Thầy Hùng
1,C2,Lý cơ bản,Cô Mai
2,C4,Hóa nâng cao,Thầy Tú



=== df_scores ===


,student_id,subject,score
0,S1,Toán,9
1,S1,Lý,8
2,S2,Toán,7
3,S3,Toán,6
4,S3,Lý,9
5,S6,Toán,10


# 1 - MERGE CƠ BẢN (Basic Merge)

## Merge là gì?

**Merge** (ghép bảng) là thao tác **kết hợp 2 DataFrame** dựa trên **cột chung** (key column).
Tương đương với **JOIN** trong SQL.

### Khi nào cần merge?
- Dữ liệu nằm ở **nhiều bảng khác nhau** (users, accounts, transactions)
- Muốn kết hợp thông tin để phân tích (ví dụ: xem thu nhập của user theo từng giao dịch)

### Cú pháp

```python
pd.merge(left, right, on='key_column', how='inner')
```

| Tham số | Mô tả |
|:---|:---|
| `left` | DataFrame bên trái |
| `right` | DataFrame bên phải |
| `on` | Tên cột chung để merge |
| `how` | Kiểu merge: `'inner'`, `'left'`, `'right'`, `'outer'` |

---

## 4 kiểu merge

| Kiểu | Mô tả | Giữ lại |
|:---|:---|:---|
| `inner` | Chỉ giữ dòng có key ở **cả 2 bảng** | Giao nhau |
| `left` | Giữ **tất cả** bên trái | Toàn bộ bảng trái + khớp bảng phải |
| `right` | Giữ **tất cả** bên phải | Toàn bộ bảng phải + khớp bảng trái |
| `outer` | Giữ **tất cả** từ cả 2 bảng | Hợp nhau |

### Minh họa (Venn Diagram)

```
    INNER:  chỉ phần chung        LEFT:  toàn bộ bên trái
    ┌───┐ ┌───┐                   ┌███┐ ┌───┐
    │   │█│   │                   │███│█│   │
    └───┘ └───┘                   └███┘ └───┘

    RIGHT: toàn bộ bên phải       OUTER: toàn bộ cả hai
    ┌───┐ ┌───┐                   ┌███┐ ┌███┐
    │   │█│███│                   │███│█│███│
    └───┘ └███┘                   └███┘ └███┘
```

## 1.1 Inner Merge (mặc định)

Chỉ giữ những dòng có **key xuất hiện ở cả 2 bảng**.

Với dữ liệu demo:
- `df_students` có class_id: C1, C1, C2, C2, **C3**
- `df_classes` có class_id: C1, C2, **C4**
- Chỉ C1 và C2 là **chung** → inner merge giữ lại các dòng có C1, C2

In [11]:
print('Classes:')
display(df_classes)

print('\nStudents:')
display(df_students)

# Inner Merge: students + classes
print('\nInner Merge:')
inner = pd.merge(
    df_students, 
    df_classes, 
    on='class_id', 
    how='inner'
)
display(inner)

# Quan sát:
# - Em (class_id=C3) bị loại → vì C3 không có trong df_classes
# - Lớp C4 (Hóa nâng cao) bị loại → vì không có học sinh nào thuộc C4
# - Kết quả: 4 dòng (S1, S2 thuộc C1; S3, S4 thuộc C2)

Class:


,class_id,class_name,teacher
0,C1,Toán nâng cao,Thầy Hùng
1,C2,Lý cơ bản,Cô Mai
2,C4,Hóa nâng cao,Thầy Tú



Students:


,student_id,name,class_id
0,S1,An,C1
1,S2,Bình,C1
2,S3,Chi,C2
3,S4,Dũng,C2
4,S5,Em,C3



Inner Merge:


,student_id,name,class_id,class_name,teacher
0,S1,An,C1,Toán nâng cao,Thầy Hùng
1,S2,Bình,C1,Toán nâng cao,Thầy Hùng
2,S3,Chi,C2,Lý cơ bản,Cô Mai
3,S4,Dũng,C2,Lý cơ bản,Cô Mai


## 1.2 Left Merge

Giữ **tất cả dòng bên trái** (df_students). Nếu không tìm thấy key ở bảng phải → điền `NaN`.

In [10]:
print('Classes:')
display(df_classes)

print('\nStudents:')
display(df_students)

# Left Merge: giữ tất cả học sinh
print('\nLeft Merge:')
left = pd.merge(df_students, 
    df_classes, 
    on='class_id', 
    how='left'
)
display(left)

# Quan sát:
# - Em (C3) VẪN xuất hiện → nhưng class_name và teacher là NaN
# - Lớp C4 vẫn bị loại (vì C4 không ở bảng trái)
# - Kết quả: 5 dòng (giữ nguyên tất cả học sinh)

Class:


,class_id,class_name,teacher
0,C1,Toán nâng cao,Thầy Hùng
1,C2,Lý cơ bản,Cô Mai
2,C4,Hóa nâng cao,Thầy Tú



Students:


,student_id,name,class_id
0,S1,An,C1
1,S2,Bình,C1
2,S3,Chi,C2
3,S4,Dũng,C2
4,S5,Em,C3



Left Merge:


,student_id,name,class_id,class_name,teacher
0,S1,An,C1,Toán nâng cao,Thầy Hùng
1,S2,Bình,C1,Toán nâng cao,Thầy Hùng
2,S3,Chi,C2,Lý cơ bản,Cô Mai
3,S4,Dũng,C2,Lý cơ bản,Cô Mai
4,S5,Em,C3,NaN,NaN


## 1.3 Right Merge

Giữ **tất cả dòng bên phải** (df_classes). Nếu không tìm thấy key ở bảng trái → điền `NaN`.

In [12]:
print('Classes:')
display(df_classes)

print('\nStudents:')
display(df_students)

# Right Merge: giữ tất cả lớp học
print('\nRight Merge:')
right = pd.merge(
    df_students, 
    df_classes, 
    on='class_id', 
    how='right'
)
display(right)

# Quan sát:
# - Lớp C4 (Hóa nâng cao) VẪN xuất hiện → nhưng student_id và name là NaN
# - Em (C3) bị loại (vì C3 không ở bảng phải)
# - Kết quả: 5 dòng

Class:


,class_id,class_name,teacher
0,C1,Toán nâng cao,Thầy Hùng
1,C2,Lý cơ bản,Cô Mai
2,C4,Hóa nâng cao,Thầy Tú



Students:


,student_id,name,class_id
0,S1,An,C1
1,S2,Bình,C1
2,S3,Chi,C2
3,S4,Dũng,C2
4,S5,Em,C3



Right Merge:


,student_id,name,class_id,class_name,teacher
0,S1,An,C1,Toán nâng cao,Thầy Hùng
1,S2,Bình,C1,Toán nâng cao,Thầy Hùng
2,S3,Chi,C2,Lý cơ bản,Cô Mai
3,S4,Dũng,C2,Lý cơ bản,Cô Mai
4,NaN,NaN,C4,Hóa nâng cao,Thầy Tú


## 1.4 Outer Merge

Giữ **tất cả dòng từ cả 2 bảng**. Dòng nào không khớp → điền `NaN`.

In [13]:
print('Classes:')
display(df_classes)

print('\nStudents:')
display(df_students)

# Outer Merge: giữ tất cả
print('\nOuter Merge:')
outer = pd.merge(
    df_students, 
    df_classes, 
    on='class_id', 
    how='outer'
)
display(outer)

# Quan sát:
# - Em (C3) xuất hiện → class_name, teacher là NaN
# - Lớp C4 xuất hiện → student_id, name là NaN
# - Kết quả: 6 dòng (nhiều nhất trong các kiểu merge)

Classes:


,class_id,class_name,teacher
0,C1,Toán nâng cao,Thầy Hùng
1,C2,Lý cơ bản,Cô Mai
2,C4,Hóa nâng cao,Thầy Tú



Students:


,student_id,name,class_id
0,S1,An,C1
1,S2,Bình,C1
2,S3,Chi,C2
3,S4,Dũng,C2
4,S5,Em,C3



Outer Merge:


,student_id,name,class_id,class_name,teacher
0,S1,An,C1,Toán nâng cao,Thầy Hùng
1,S2,Bình,C1,Toán nâng cao,Thầy Hùng
2,S3,Chi,C2,Lý cơ bản,Cô Mai
3,S4,Dũng,C2,Lý cơ bản,Cô Mai
4,S5,Em,C3,NaN,NaN
5,NaN,NaN,C4,Hóa nâng cao,Thầy Tú


## So sánh kết quả

| Kiểu merge | Số dòng | Em (C3) | Lớp C4 |
|:---|:---|:---|:---|
| `inner` | 4 | ❌ Bị loại | ❌ Bị loại |
| `left` | 5 | ✅ Giữ lại (NaN) | ❌ Bị loại |
| `right` | 5 | ❌ Bị loại | ✅ Giữ lại (NaN) |
| `outer` | 6 | ✅ Giữ lại (NaN) | ✅ Giữ lại (NaN) |

## 1.5 Áp dụng với dữ liệu Kaggle

Merge `users` và `accounts` trên cột `user_id`:

In [14]:
# Inner merge: users + accounts
user_accounts = pd.merge(
    users, 
    accounts, 
    on='user_id', 
    how='inner'
)
print(f"users: {users.shape[0]} dòng")
print(f"accounts: {accounts.shape[0]} dòng")
print(f"Sau inner merge: {user_accounts.shape[0]} dòng, {user_accounts.shape[1]} cột")
display(user_accounts.head())

users: 5000 dòng
accounts: 7468 dòng
Sau inner merge: 7468 dòng, 13 cột


,user_id,age,employment_type,income,credit_score,risk_profile,home_location,account_created_at,account_id,account_type,balance,kyc_verified,account_age_days
0,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A1,savings,244465,1,782
1,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A2,current,456249,0,782
2,U2,62,student,10066,328,high,Mumbai,2021-04-12 22:39:56.896382,A3,savings,22496,0,1775
3,U2,62,student,10066,328,high,Mumbai,2021-04-12 22:39:56.896382,A4,current,29664,0,1775
4,U3,59,salaried,75331,547,high,Hyderabad,2021-03-09 22:39:56.896382,A5,savings,78744,0,1809


In [15]:
# Inner merge: accounts + transactions
acct_txns = pd.merge(
    accounts, 
    transactions, 
    on='account_id', 
    how='inner'
)
print(f"accounts: {accounts.shape[0]} dòng")
print(f"transactions: {transactions.shape[0]} dòng")
print(f"Sau inner merge: {acct_txns.shape[0]} dòng, {acct_txns.shape[1]} cột")
display(acct_txns.head())

accounts: 7468 dòng
transactions: 221514 dòng
Sau inner merge: 221514 dòng, 15 cột


,account_id,user_id_x,account_type,balance,kyc_verified,account_age_days,txn_id,user_id_y,amount,merchant_category,txn_time,txn_location,is_international,status,is_fraud
0,A1,U1,savings,244465,1,782,T3,U1,2319.51,entertainment,2026-02-15 21:39:58.249568,Bangalore,1,success,0
1,A1,U1,savings,244465,1,782,T5,U1,3727.33,travel,2026-01-11 11:39:58.249568,Mumbai,0,success,0
2,A1,U1,savings,244465,1,782,T6,U1,3099.61,groceries,2026-01-02 21:39:58.249568,Delhi,0,success,0
3,A1,U1,savings,244465,1,782,T7,U1,6105.44,electronics,2026-01-09 21:39:58.249568,Kanpur,0,success,0
4,A1,U1,savings,244465,1,782,T11,U1,1481.86,groceries,2025-12-22 11:39:58.249568,Kanpur,0,success,0


## 1.6 `on` vs `left_on` / `right_on`

Khi **tên cột khác nhau** giữa 2 bảng, dùng `left_on` và `right_on`:

```python
# Khi cột chung cùng tên
pd.merge(df_A, df_B, on='id')

# Khi cột chung khác tên
pd.merge(df_A, df_B, left_on='ma_sv', right_on='student_id')
```

In [17]:
# Ví dụ: 2 bảng có tên cột khác nhau
df_A = pd.DataFrame({'ma_sv': [1, 2, 3], 'ten': ['An', 'Bình', 'Chi']})
df_B = pd.DataFrame({'student_id': [1, 2, 4], 'diem': [9, 8, 7]})

print("df_A:")
display(df_A)
print("df_B:")
display(df_B)

# Merge với left_on / right_on
result = pd.merge(
    df_A, 
    df_B, 
    left_on='ma_sv', 
    right_on='student_id', 
    how='inner'
)
print("\nKết quả merge:")
display(result)

df_A:


,ma_sv,ten
0,1,An
1,2,Bình
2,3,Chi


df_B:


,student_id,diem
0,1,9
1,2,8
2,4,7



Kết quả merge:


,ma_sv,ten,student_id,diem
0,1,An,1,9
1,2,Bình,2,8


## Bài tập — Merge Cơ Bản

In [ ]:
# @title Bài tập 1
# Merge users và accounts (inner merge) trên cột user_id
# Đếm số dòng kết quả


In [ ]:
# @title Bài tập 2
# Merge accounts và transactions (left merge) trên cột account_id
# Có account nào không có giao dịch không? (kiểm tra NaN ở cột txn_id)


In [ ]:
# @title Bài tập 3
# Merge users và accounts (outer merge)
# Tìm những user_id không có tài khoản (lọc dòng có account_id là NaN)


In [ ]:
# @title Bài tập 4
# Merge accounts và transactions (inner merge)
# Sau đó tính tổng amount theo account_type


# 2 - MERGE NÂNG CAO (Advanced Merge)

## Các kỹ thuật nâng cao

| Kỹ thuật | Mô tả | Khi nào dùng |
|:---|:---|:---|
| **Suffixes** | Xử lý cột trùng tên | 2 bảng có cột cùng tên (không phải key) |
| **Indicator** | Thêm cột `_merge` | Muốn biết dòng đến từ bảng nào |
| **Merge 3+ bảng** | Merge liên tiếp | Cần kết hợp nhiều bảng |
| **Validate** | Kiểm tra quan hệ | Đảm bảo dữ liệu đúng cấu trúc |

## 2.1 Suffixes — Xử lý cột trùng tên

Khi 2 bảng có **cột trùng tên** (không phải key), pandas tự thêm hậu tố `_x`, `_y` để phân biệt.

Có thể tùy chỉnh bằng tham số `suffixes=('_left', '_right')`.

In [18]:
# Tạo 2 bảng có cột trùng tên "location"
df_user_loc = users[['user_id', 'home_location']].head(5).copy()
df_user_loc.columns = ['user_id', 'location']

df_acct_loc = pd.DataFrame({
    'user_id': users['user_id'].head(5),
    'location': ['HCM', 'HN', 'DN', 'HCM', 'HN']  # địa chỉ giao dịch
})

print("df_user_loc (địa chỉ nhà):")
display(df_user_loc)
print("df_acct_loc (địa chỉ giao dịch):")
display(df_acct_loc)

# Merge → cột location bị trùng
merged = pd.merge(
    df_user_loc, 
    df_acct_loc, 
    on='user_id'
)
print("\nMerge mặc định (suffixes _x, _y):")
display(merged)

# Tùy chỉnh suffixes
merged_custom = pd.merge(
    df_user_loc, 
    df_acct_loc, 
    on='user_id',
    suffixes=('_home', '_txn')
)
print("\nMerge với suffixes tùy chỉnh:")
display(merged_custom)

df_user_loc (địa chỉ nhà):


,user_id,location
0,U1,Mumbai
1,U2,Mumbai
2,U3,Hyderabad
3,U4,Delhi
4,U5,Bangalore


df_acct_loc (địa chỉ giao dịch):


,user_id,location
0,U1,HCM
1,U2,HN
2,U3,DN
3,U4,HCM
4,U5,HN



Merge mặc định (suffixes _x, _y):


,user_id,location_x,location_y
0,U1,Mumbai,HCM
1,U2,Mumbai,HN
2,U3,Hyderabad,DN
3,U4,Delhi,HCM
4,U5,Bangalore,HN



Merge với suffixes tùy chỉnh:


,user_id,location_home,location_txn
0,U1,Mumbai,HCM
1,U2,Mumbai,HN
2,U3,Hyderabad,DN
3,U4,Delhi,HCM
4,U5,Bangalore,HN


## 2.2 Indicator — Kiểm tra nguồn dữ liệu

Thêm `indicator=True` để tạo cột `_merge` cho biết mỗi dòng đến từ đâu:

| Giá trị `_merge` | Ý nghĩa |
|:---|:---|
| `both` | Có ở cả 2 bảng |
| `left_only` | Chỉ có ở bảng trái |
| `right_only` | Chỉ có ở bảng phải |

Rất hữu ích để **tìm dữ liệu không khớp**.

In [19]:
# Indicator: tìm học sinh không có lớp và lớp không có học sinh
result = pd.merge(
    df_students, 
    df_classes, 
    on='class_id', 
    how='outer', 
    indicator=True
)
display(result)

print("\nThống kê _merge:")
display(result['_merge'].value_counts())

,student_id,name,class_id,class_name,teacher,_merge
0,S1,An,C1,Toán nâng cao,Thầy Hùng,both
1,S2,Bình,C1,Toán nâng cao,Thầy Hùng,both
2,S3,Chi,C2,Lý cơ bản,Cô Mai,both
3,S4,Dũng,C2,Lý cơ bản,Cô Mai,both
4,S5,Em,C3,NaN,NaN,left_only
5,NaN,NaN,C4,Hóa nâng cao,Thầy Tú,right_only



Thống kê _merge:


_merge
both          4
left_only     1
right_only    1
Name: count, dtype: int64

In [20]:
# Ứng dụng: Tìm user KHÔNG có tài khoản
check = pd.merge(
    users, 
    accounts, 
    on='user_id', 
    how='left', 
    indicator=True
)

no_account = check[check['_merge'] == 'left_only']
print(f"Số user không có tài khoản: {len(no_account)}")
display(no_account[['user_id', 'age', 'income', '_merge']].head())

Số user không có tài khoản: 0


,user_id,age,income,_merge


## 2.3 Merge 3 bảng

Để tạo bảng tổng hợp **users → accounts → transactions**, merge **tuần tự từng cặp**:

```python
# Bước 1: users + accounts (on user_id)
# Bước 2: kết quả bước 1 + transactions (on account_id)
```

In [21]:
# Merge 3 bảng: users → accounts → transactions
# Bước 1: users + accounts
step1 = pd.merge(
    users, 
    accounts, 
    on='user_id', 
    how='inner'
)
print(f"Bước 1 (users + accounts): {step1.shape}")

# Bước 2: step1 + transactions
full_data = pd.merge(
    step1, 
    transactions, 
    on='account_id', 
    how='inner'
)
print(f"Bước 2 (+ transactions): {full_data.shape}")

print("\nCác cột trong bảng tổng hợp:")
print(full_data.columns.tolist())
display(full_data.head())

Bước 1 (users + accounts): (7468, 13)
Bước 2 (+ transactions): (221514, 22)

Các cột trong bảng tổng hợp:
['user_id_x', 'age', 'employment_type', 'income', 'credit_score', 'risk_profile', 'home_location', 'account_created_at', 'account_id', 'account_type', 'balance', 'kyc_verified', 'account_age_days', 'txn_id', 'user_id_y', 'amount', 'merchant_category', 'txn_time', 'txn_location', 'is_international', 'status', 'is_fraud']


,user_id_x,age,employment_type,income,credit_score,risk_profile,home_location,account_created_at,account_id,account_type,...,account_age_days,txn_id,user_id_y,amount,merchant_category,txn_time,txn_location,is_international,status,is_fraud
0,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A1,savings,...,782,T3,U1,2319.51,entertainment,2026-02-15 21:39:58.249568,Bangalore,1,success,0
1,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A1,savings,...,782,T5,U1,3727.33,travel,2026-01-11 11:39:58.249568,Mumbai,0,success,0
2,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A1,savings,...,782,T6,U1,3099.61,groceries,2026-01-02 21:39:58.249568,Delhi,0,success,0
3,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A1,savings,...,782,T7,U1,6105.44,electronics,2026-01-09 21:39:58.249568,Kanpur,0,success,0
4,U1,40,self-employed,112942,611,medium,Mumbai,2023-12-31 22:39:56.896382,A1,savings,...,782,T11,U1,1481.86,groceries,2025-12-22 11:39:58.249568,Kanpur,0,success,0


In [22]:
# Phân tích trên bảng tổng hợp:
# Amount trung bình theo risk_profile
print("Amount trung bình theo risk_profile:")
display(full_data.groupby('risk_profile')['amount'].mean())

# Tỷ lệ fraud theo risk_profile
print("\nTỷ lệ fraud theo risk_profile:")
display(full_data.groupby('risk_profile')['is_fraud'].mean())

Amount trung bình theo risk_profile:


risk_profile
high       2689.858260
low       10693.328746
medium     7315.828871
Name: amount, dtype: float64


Tỷ lệ fraud theo risk_profile:


risk_profile
high      0.045572
low       0.046103
medium    0.046516
Name: is_fraud, dtype: float64

## 2.4 Validate — Kiểm tra quan hệ dữ liệu

Tham số `validate` giúp kiểm tra quan hệ giữa 2 bảng khi merge:

| Giá trị | Ý nghĩa |
|:---|:---|
| `'one_to_one'` / `'1:1'` | Mỗi key chỉ xuất hiện 1 lần ở cả 2 bảng |
| `'one_to_many'` / `'1:m'` | Key bên trái là duy nhất, bên phải có thể lặp |
| `'many_to_one'` / `'m:1'` | Key bên phải là duy nhất, bên trái có thể lặp |
| `'many_to_many'` / `'m:m'` | Không kiểm tra (mặc định) |

Nếu dữ liệu vi phạm → **báo lỗi** → giúp phát hiện lỗi sớm.

In [23]:
# validate: users (1) → accounts (many) trên user_id
# Mỗi user có thể có nhiều account → one_to_many
try:
    result = pd.merge(
        users, 
        accounts, 
        on='user_id', 
        how='inner', 
        validate='one_to_many'
    )
    print("✅ Validate one_to_many: PASSED")
    print(f"Kết quả: {result.shape[0]} dòng")
except Exception as e:
    print(f"❌ Validate FAILED: {e}")

✅ Validate one_to_many: PASSED
Kết quả: 7468 dòng


In [24]:
# validate sẽ báo lỗi nếu quan hệ sai
# Ví dụ: accounts → users KHÔNG phải one_to_one (vì 1 user có nhiều account)
try:
    result = pd.merge(
        accounts, 
        users, 
        on='user_id', 
        how='inner', 
        validate='one_to_one'
    )
    print("✅ Validate one_to_one: PASSED")
except Exception as e:
    print(f"❌ Validate FAILED: {e}")
    print("→ Vì 1 user_id xuất hiện nhiều lần trong accounts (1 user có nhiều tài khoản)")

❌ Validate FAILED: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
 user_id
     U1
     U2
     U5
     U7
     U9 ...
→ Vì 1 user_id xuất hiện nhiều lần trong accounts (1 user có nhiều tài khoản)


## Bài tập — Merge Nâng Cao

In [ ]:
# @title Bài tập 1
# Merge 3 bảng: users → accounts → transactions
# Tính số tiền giao dịch trung bình (amount) theo home_location


In [ ]:
# @title Bài tập 2
# Dùng indicator để tìm account KHÔNG có giao dịch nào
# Gợi ý: merge accounts và transactions với how='left', indicator=True
#         lọc dòng _merge == 'left_only'


In [ ]:
# @title Bài tập 3
# Merge full_data (3 bảng đã merge)
# Tìm nhóm risk_profile có tỷ lệ giao dịch quốc tế (is_international) cao nhất


In [ ]:
# @title Bài tập 4
# Merge users + accounts
# Tính balance trung bình theo (risk_profile, account_type)
# Dùng groupby sau merge


# 3 - QUERY (Truy vấn dữ liệu)

`df.query()` là cách viết **filter dạng chuỗi** — gọn hơn boolean indexing, đặc biệt khi điều kiện phức tạp.

### Tại sao cần `query()`?

Khi dữ liệu đã merge (nhiều cột), boolean indexing trở nên **dài và khó đọc**:

```python
# Boolean indexing — dài dòng:
full_data[(full_data['risk_profile'] == 'high') & 
          (full_data['amount'] > 10000) & 
          (full_data['is_international'] == 1)]

# query() — ngắn gọn, đọc như câu tiếng Anh:
full_data.query('risk_profile == "high" and amount > 10000 and is_international == 1')
```

### Cú pháp cơ bản

| Cú pháp | Ví dụ |
|:---|:---|
| So sánh | `df.query('age > 30')` |
| Bằng (string) | `df.query('city == "Mumbai"')` |
| AND | `df.query('age > 30 and income > 50000')` |
| OR | `df.query('age > 30 or income > 50000')` |
| NOT | `df.query('not is_fraud')` |
| IN | `df.query('city in ["Mumbai", "Delhi"]')` |
| Biến ngoài `@` | `df.query('age > @min_age')` |

## 3.1 Query cơ bản — So sánh với Boolean Indexing

In [ ]:
# === So sánh 2 cách viết ===

# Cách 1: Boolean indexing (đã học ở Chap 1)
result1 = users[users['age'] > 40]
print(f"Boolean indexing: {len(result1)} dòng")

# Cách 2: query()
result2 = users.query('age > 40')
print(f"query():          {len(result2)} dòng")

# Kết quả giống nhau!
display(result2.head())

## 3.2 Query nhiều điều kiện (AND / OR / NOT)

In [ ]:
# AND — Khách hàng tuổi > 40 VÀ risk_profile = 'high'

# Boolean indexing: dài, nhiều ngoặc
r1 = users[(users['age'] > 40) & (users['risk_profile'] == 'high')]

# query(): gọn gàng
r2 = users.query('age > 40 and risk_profile == "high"')

print(f"Boolean indexing: {len(r1)} dòng")
print(f"query():          {len(r2)} dòng")
display(r2.head())

In [ ]:
# OR — Khách hàng ở Mumbai HOẶC Delhi
result = users.query('home_location == "Mumbai" or home_location == "Delhi"')
print(f"Mumbai hoặc Delhi: {len(result)} dòng")

# Cách viết gọn hơn với IN
result2 = users.query('home_location in ["Mumbai", "Delhi"]')
print(f"Dùng IN:           {len(result2)} dòng")
display(result2.head())

In [ ]:
# NOT — Giao dịch KHÔNG phải fraud
non_fraud = transactions.query('not is_fraud')
print(f"Giao dịch không fraud: {len(non_fraud)}")

# NOT IN — Giao dịch KHÔNG phải ở Mumbai và Delhi
other_cities = transactions.query('txn_location not in ["Mumbai", "Delhi"]')
print(f"Giao dịch ngoài Mumbai/Delhi: {len(other_cities)}")

## 3.3 Tham chiếu biến bên ngoài với `@`

Dùng `@tên_biến` để sử dụng biến Python bên trong chuỗi query.

In [ ]:
# Sử dụng biến bên ngoài với @
min_age = 30
max_age = 50
min_income = 80000

result = users.query('@min_age <= age <= @max_age and income > @min_income')
print(f"Users tuổi {min_age}-{max_age}, income > {min_income}: {len(result)} dòng")
display(result.head())

In [ ]:
# Ứng dụng: Lọc giao dịch lớn hơn trung bình
avg_amount = transactions['amount'].mean()
print(f"Amount trung bình: {avg_amount:.2f}")

big_txns = transactions.query('amount > @avg_amount')
print(f"Giao dịch lớn hơn TB: {len(big_txns)} / {len(transactions)} ({len(big_txns)/len(transactions)*100:.1f}%)")

## 3.4 Query trên dữ liệu đã merge

`query()` thực sự tỏa sáng khi làm việc với bảng đã merge (nhiều cột, điều kiện phức tạp).

In [ ]:
# Nhắc lại: full_data đã được merge ở Section 2.3
print(f"full_data: {full_data.shape[0]} dòng, {full_data.shape[1]} cột")
print(f"Các cột: {full_data.columns.tolist()}")

In [ ]:
# Ví dụ 1: Giao dịch fraud của khách hàng risk_profile 'high' với amount > 10000

# Boolean indexing — rất dài:
r1 = full_data[(full_data['risk_profile'] == 'high') & 
               (full_data['is_fraud'] == 1) & 
               (full_data['amount'] > 10000)]

# query() — gọn gàng:
r2 = full_data.query('risk_profile == "high" and is_fraud == 1 and amount > 10000')

print(f"Kết quả: {len(r2)} giao dịch")
display(r2[['user_id', 'risk_profile', 'amount', 'is_fraud', 'merchant_category']].head())

In [ ]:
# Ví dụ 2: Giao dịch quốc tế tại Travel hoặc Shopping, amount > 5000
result = full_data.query(
    'is_international == 1 and merchant_category in ["travel", "shopping"] and amount > 5000'
)
print(f"Giao dịch quốc tế Travel/Shopping > 5000: {len(result)} giao dịch")
display(result[['user_id', 'merchant_category', 'amount', 'txn_location']].head())

In [ ]:
# Ví dụ 3: Kết hợp query() + groupby()
# Chỉ lấy khách hàng risk 'high' → tính amount TB theo merchant
full_data.query('risk_profile == "high"').groupby('merchant_category')['amount'].mean()

## Bài tập — Query

In [ ]:
# @title Bài tập 1
# Dùng query() lọc users có credit_score > 600 và income > 100000
# Hiển thị 5 dòng đầu


In [ ]:
# @title Bài tập 2
# Dùng query() lọc transactions: giao dịch fraud, amount > 20000
# Đếm số lượng và hiển thị 5 dòng đầu


In [ ]:
# @title Bài tập 3
# Tạo biến threshold = 50000
# Dùng query() với @ để lọc transactions có amount > threshold
# Tính tỷ lệ fraud trong nhóm này


In [ ]:
# @title Bài tập 4
# Trên full_data, dùng query() lọc:
#   - risk_profile là 'high' hoặc 'medium'
#   - is_international == 1
#   - amount > 10000
# Sau đó dùng groupby('merchant_category') tính amount trung bình


# 4 - PIVOT (Chuyển đổi dạng dữ liệu)

## Pivot là gì?

**Pivot** = chuyển dữ liệu từ dạng **"dài" (long format)** sang dạng **"rộng" (wide format)**.

### Ví dụ minh họa

**Dạng dài** (long):

| Học sinh | Môn | Điểm |
|:---|:---|:---|
| An | Toán | 9 |
| An | Lý | 8 |
| Bình | Toán | 7 |

**Dạng rộng** (wide) — sau pivot:

| Học sinh | Toán | Lý |
|:---|:---|:---|
| An | 9 | 8 |
| Bình | 7 | NaN |

### Cú pháp

```python
df.pivot(index='Học sinh', columns='Môn', values='Điểm')
```

| Tham số | Mô tả |
|:---|:---|
| `index` | Cột làm **hàng** (index) |
| `columns` | Cột làm **tên cột mới** |
| `values` | Cột chứa **giá trị** điền vào bảng |

### ⚠️ Lưu ý quan trọng
`pivot()` sẽ **báo lỗi** nếu có **giá trị trùng lặp** (ví dụ: An có 2 điểm Toán).
Khi đó phải dùng `pivot_table()` (phần 4).

## 5.1 Pivot với dữ liệu demo

In [27]:
# Tạo bảng điểm (không trùng lặp)
df_diem = pd.DataFrame({
    'student': ['An', 'An', 'An', 'Bình', 'Bình', 'Bình', 'Chi', 'Chi', 'Chi'],
    'subject': ['Toán', 'Lý', 'Hóa', 'Toán', 'Lý', 'Hóa', 'Toán', 'Lý', 'Hóa'],
    'score': [9, 8, 7, 7, 6, 8, 8, 9, 7]
})

print("Dạng dài (long format):")
display(df_diem)

# Pivot → dạng rộng
pivot_diem = df_diem.pivot(
    index='student', 
    columns='subject', 
    values='score'
)
print("\nDạng rộng (wide format) — sau pivot:")
display(pivot_diem)

Dạng dài (long format):


,student,subject,score
0,An,Toán,9
1,An,Lý,8
2,An,Hóa,7
3,Bình,Toán,7
4,Bình,Lý,6
5,Bình,Hóa,8
6,Chi,Toán,8
7,Chi,Lý,9
8,Chi,Hóa,7



Dạng rộng (wide format) — sau pivot:


subject,Hóa,Lý,Toán
student,,,
An,7,8,9
Bình,8,6,7
Chi,7,9,8


## 5.2 Pivot dữ liệu doanh thu

In [28]:
# Bảng doanh thu theo tháng và sản phẩm
df_sales = pd.DataFrame({
    'month': ['T1', 'T1', 'T1', 'T2', 'T2', 'T2', 'T3', 'T3', 'T3'],
    'product': ['Laptop', 'Phone', 'Tablet', 'Laptop', 'Phone', 'Tablet', 'Laptop', 'Phone', 'Tablet'],
    'revenue': [500, 300, 200, 550, 320, 180, 600, 350, 220]
})

print("Dạng dài:")
display(df_sales)

# Pivot: xem doanh thu theo sản phẩm mỗi tháng
pivot_sales = df_sales.pivot(
    index='month', 
    columns='product', 
    values='revenue'
)
print("\nSau pivot (doanh thu theo sản phẩm / tháng):")
display(pivot_sales)

Dạng dài:


,month,product,revenue
0,T1,Laptop,500
1,T1,Phone,300
2,T1,Tablet,200
3,T2,Laptop,550
4,T2,Phone,320
5,T2,Tablet,180
6,T3,Laptop,600
7,T3,Phone,350
8,T3,Tablet,220



Sau pivot (doanh thu theo sản phẩm / tháng):


product,Laptop,Phone,Tablet
month,,,
T1,500,300,200
T2,550,320,180
T3,600,350,220


## 5.3 Khi pivot() bị lỗi — Giá trị trùng lặp

Nếu có **nhiều giá trị** cho cùng một cặp (index, columns), `pivot()` sẽ báo lỗi.

Ví dụ: Nếu An có **2 điểm Toán** → pivot không biết chọn điểm nào.

In [32]:
df_dup = pd.DataFrame({
    'student': ['An', 'An', 'An', 'Bình', 'Bình'],
    'subject': ['Toán', 'Toán', 'Lý',   'Toán', 'Lý'],   # An-Toán xuất hiện 2 lần!
    'score':   [9,     7,     8,    7,      6]
})


print("Dữ liệu có trùng lặp (An thi Toán 2 lần):")
display(df_dup)

# Thử pivot → SẼ LỖI vì cặp (An, Toán) có 2 giá trị: 9 và 7
try:
    df_dup.pivot(
        index='student', 
        columns='subject', 
        values='score'
    )
except ValueError as e:
    print(f"\n❌ Lỗi: {e}")
    print("\n→ Giải pháp: Dùng pivot_table() với aggfunc để tổng hợp!")
    print("   Ví dụ: aggfunc='mean' → lấy trung bình 2 lần thi")
    pt_fix = df_dup.pivot_table(
        index='student', 
        columns='subject', 
        values='score', 
        aggfunc='mean'
    )
    print("\n✅ pivot_table() với aggfunc='mean':")
    display(pt_fix)

Dữ liệu có trùng lặp (An thi Toán 2 lần):


,student,subject,score
0,An,Toán,9
1,An,Toán,7
2,An,Lý,8
3,Bình,Toán,7
4,Bình,Lý,6



❌ Lỗi: Index contains duplicate entries, cannot reshape

→ Giải pháp: Dùng pivot_table() với aggfunc để tổng hợp!
   Ví dụ: aggfunc='mean' → lấy trung bình 2 lần thi

✅ pivot_table() với aggfunc='mean':


subject,Lý,Toán
student,,
An,8.0,8.0
Bình,6.0,7.0


## Bài tập — Pivot

In [ ]:
# @title Bài tập 1
# Tạo DataFrame sau và pivot nó:
# df_temp = pd.DataFrame({
#     'city': ['HN','HN','HN','HCM','HCM','HCM'],
#     'month': ['T1','T2','T3','T1','T2','T3'],
#     'temp': [18, 22, 28, 25, 30, 32]
# })
# Pivot: index='city', columns='month', values='temp'


In [ ]:
# @title Bài tập 2
# Từ full_data (3 bảng đã merge), tạo bảng tổng hợp:
# 1. Dùng groupby để tính amount trung bình theo (merchant_category, account_type)
# 2. reset_index() kết quả
# 3. Pivot kết quả: index='merchant_category', columns='account_type', values='amount'


# 5 - PIVOT TABLE (Bảng tổng hợp)

## Pivot Table là gì?

**Pivot Table** = `pivot()` + **aggregation** (tổng hợp).

### So sánh pivot() vs pivot_table()

| | `pivot()` | `pivot_table()` |
|:---|:---|:---|
| Dữ liệu trùng lặp | ❌ Báo lỗi | ✅ Tổng hợp bằng aggfunc |
| Hàm tổng hợp | Không có | `mean`, `sum`, `count`, `min`, `max`, ... |
| Tổng cộng (margins) | Không có | ✅ `margins=True` |
| Thay NaN | Không có | ✅ `fill_value` |
| Nhiều cấp index | Hạn chế | ✅ Hỗ trợ đầy đủ |

### Cú pháp

```python
pd.pivot_table(
    df,
    values='amount',           # Cột chứa giá trị
    index='category',          # Cột làm hàng
    columns='type',            # Cột làm cột
    aggfunc='mean',            # Hàm tổng hợp
    margins=True,              # Thêm tổng cộng
    fill_value=0               # Thay NaN bằng 0
)
```

### Các `aggfunc` thường dùng

| aggfunc | Mô tả |
|:---|:---|
| `'mean'` | Trung bình (mặc định) |
| `'sum'` | Tổng |
| `'count'` | Đếm số lượng |
| `'min'` / `'max'` | Giá trị nhỏ nhất / lớn nhất |
| `'median'` | Trung vị |
| `['mean', 'sum']` | Nhiều hàm cùng lúc |

## 5.1 Pivot Table cơ bản

In [33]:
# Tạo dữ liệu đơn hàng (có trùng lặp month + product)
df_orders = pd.DataFrame({
    'month': ['T1','T1','T1','T1','T2','T2','T2','T2','T3','T3','T3','T3'],
    'product': ['Laptop','Laptop','Phone','Phone','Laptop','Laptop','Phone','Phone','Laptop','Laptop','Phone','Phone'],
    'region': ['Bắc','Nam','Bắc','Nam','Bắc','Nam','Bắc','Nam','Bắc','Nam','Bắc','Nam'],
    'revenue': [500, 450, 300, 280, 520, 470, 310, 290, 600, 550, 350, 320],
    'quantity': [5, 4, 10, 9, 6, 5, 11, 10, 7, 6, 12, 11]
})

print("Dữ liệu đơn hàng:")
display(df_orders)

# pivot() sẽ lỗi vì T1-Laptop xuất hiện 2 lần (Bắc + Nam)
# pivot_table() tự động tổng hợp!

pt = pd.pivot_table(
    df_orders, 
    values='revenue', 
    index='month', 
    columns='product', 
    aggfunc='mean'
)
print("\nPivot Table — Doanh thu trung bình:")
display(pt)

Dữ liệu đơn hàng:


,month,product,region,revenue,quantity
0,T1,Laptop,Bắc,500,5
1,T1,Laptop,Nam,450,4
2,T1,Phone,Bắc,300,10
3,T1,Phone,Nam,280,9
4,T2,Laptop,Bắc,520,6
5,T2,Laptop,Nam,470,5
6,T2,Phone,Bắc,310,11
7,T2,Phone,Nam,290,10
8,T3,Laptop,Bắc,600,7
9,T3,Laptop,Nam,550,6



Pivot Table — Doanh thu trung bình:


product,Laptop,Phone
month,,
T1,475.0,290.0
T2,495.0,300.0
T3,575.0,335.0


## 5.2 Các aggfunc khác nhau

In [34]:
# aggfunc='sum' — Tổng doanh thu
pt_sum = pd.pivot_table(
    df_orders, 
    values='revenue', 
    index='month', 
    columns='product', 
    aggfunc='sum'
)
print("Tổng doanh thu theo sản phẩm / tháng:")
display(pt_sum)

# aggfunc='count' — Số đơn hàng
pt_count = pd.pivot_table(
    df_orders, 
    values='revenue', 
    index='month', 
    columns='product', 
    aggfunc='count'
)
print("\nSố đơn hàng:")
display(pt_count)

# aggfunc='max' — Doanh thu cao nhất
pt_max = pd.pivot_table(
    df_orders, 
    values='revenue', 
    index='month', 
    columns='product', 
    aggfunc='max'
)
print("\nDoanh thu cao nhất:")
display(pt_max)

Tổng doanh thu theo sản phẩm / tháng:


product,Laptop,Phone
month,,
T1,950,580
T2,990,600
T3,1150,670



Số đơn hàng:


product,Laptop,Phone
month,,
T1,2,2
T2,2,2
T3,2,2



Doanh thu cao nhất:


product,Laptop,Phone
month,,
T1,500,300
T2,520,310
T3,600,350


## 5.3 Áp dụng với dữ liệu Kaggle

In [35]:
# Amount trung bình theo merchant_category và is_fraud
pt_fraud = pd.pivot_table(
    transactions,
    values='amount',
    index='merchant_category',
    columns='is_fraud',
    aggfunc='mean'
)
print("Amount trung bình theo merchant_category và is_fraud:")
display(pt_fraud)

Amount trung bình theo merchant_category và is_fraud:


is_fraud,0,1
merchant_category,,
electronics,3657.774627,20629.351199
entertainment,3620.995034,21184.768833
food,3639.207005,20547.524384
groceries,3611.058504,22070.598149
shopping,3605.496634,21965.748850
travel,3639.124251,21606.322244


In [36]:
# Số giao dịch theo merchant_category và is_international
pt_intl = pd.pivot_table(
    transactions,
    values='txn_id',
    index='merchant_category',
    columns='is_international',
    aggfunc='count'
)
print("Số giao dịch theo merchant_category và is_international:")
display(pt_intl)

Số giao dịch theo merchant_category và is_international:


is_international,0,1
merchant_category,,
electronics,32217,4624
entertainment,32532,4825
food,32070,4649
groceries,32243,4724
shopping,32034,4795
travel,31941,4860


## 5.4 `margins=True` — Thêm tổng cộng

Thêm hàng và cột **All** chứa giá trị tổng hợp của toàn bộ.

In [37]:
# Pivot table với margins
pt_margins = pd.pivot_table(
    transactions,
    values='amount',
    index='merchant_category',
    columns='is_fraud',
    aggfunc='mean',
    margins=True
)
print("Pivot Table với margins (hàng/cột All):")
display(pt_margins)

# Quan sát:
# - Cột All: trung bình amount của mỗi merchant (không phân biệt fraud)
# - Hàng All: trung bình amount theo is_fraud (không phân biệt merchant)

Pivot Table với margins (hàng/cột All):


is_fraud,0,1,All
merchant_category,,,
electronics,3657.774627,20629.351199,4430.780395
entertainment,3620.995034,21184.768833,4447.066468
food,3639.207005,20547.524384,4396.233988
groceries,3611.058504,22070.598149,4470.443029
shopping,3605.496634,21965.748850,4434.048568
travel,3639.124251,21606.322244,4462.272964
All,3628.912192,21337.864953,4440.194786


## 5.5 `fill_value` — Thay NaN

Khi một cặp (index, columns) không có dữ liệu → kết quả là `NaN`.
Dùng `fill_value` để thay bằng giá trị khác (thường là 0).

In [41]:
# Tạo dữ liệu có cặp bị thiếu
df_sparse = pd.DataFrame({
    'city': ['HN', 'HN', 'HCM', 'DN'],
    'product': ['A', 'B', 'A', 'C'],
    'sales': [100, 200, 150, 300]
})

print("Dữ liệu (không phải mọi thành phố bán mọi sản phẩm):")
display(df_sparse)

# Pivot table không fill_value
pt1 = pd.pivot_table(
    df_sparse, 
    values='sales', 
    index='city', 
    columns='product', 
    aggfunc='sum'
)
print("\nKhông fill_value (có NaN):")
display(pt1)

# Pivot table có fill_value
pt2 = pd.pivot_table(
    df_sparse, 
    values='sales', 
    index='city', 
    columns='product', 
    aggfunc='sum', 
    fill_value= 0
)
print("\nfill_value=0:")
display(pt2)

Dữ liệu (không phải mọi thành phố bán mọi sản phẩm):


,city,product,sales
0,HN,A,100
1,HN,B,200
2,HCM,A,150
3,DN,C,300



Không fill_value (có NaN):


product,A,B,C
city,,,
DN,NaN,NaN,300.0
HCM,150.0,NaN,NaN
HN,100.0,200.0,NaN



fill_value=0:


product,A,B,C
city,,,
DN,0,0,300
HCM,150,0,0
HN,100,200,0


## 5.6 Nhiều aggfunc cùng lúc

In [42]:
# Tính mean VÀ sum cùng lúc
pt_multi = pd.pivot_table(
    transactions,
    values='amount',
    index='merchant_category',
    aggfunc=['mean', 'sum', 'count']
)
print("Nhiều aggfunc cùng lúc:")
display(pt_multi)

Nhiều aggfunc cùng lúc:


,mean,sum,count
,amount,amount,amount
merchant_category,,,
electronics,4430.780395,1.632344e+08,36841
entertainment,4447.066468,1.661291e+08,37357
food,4396.233988,1.614253e+08,36719
groceries,4470.443029,1.652589e+08,36967
shopping,4434.048568,1.633016e+08,36829
travel,4462.272964,1.642161e+08,36801


## 5.7 Pivot Table nhiều cấp index

In [43]:
# Pivot table 2 cấp index: merchant_category + is_international
pt_multi_idx = pd.pivot_table(
    transactions,
    values='amount',
    index=['merchant_category', 'is_international'],
    columns='is_fraud',
    aggfunc='mean'
)
print("Pivot Table 2 cấp index:")
display(pt_multi_idx)

Pivot Table 2 cấp index:


is_fraud                                      0             1
merchant_category is_international                           
electronics       0                 3654.941776   3872.477476
                  1                 3683.722921  28049.634919
entertainment     0                 3625.259582   3714.233390
                  1                 3583.111358  28812.950295
food              0                 3634.589001   3698.996913
                  1                 3680.605051  28233.079431
groceries         0                 3608.571194   3649.253424
                  1                 3633.748598  29175.126428
shopping          0                 3594.837833   3478.308351
                  1                 3698.959514  29190.547774
travel            0                 3641.055252   3609.990854
                  1                 3622.559041  29021.895983

## 5.8 `pd.crosstab()` — Bảng chéo tần suất

`pd.crosstab()` là cách nhanh để tạo **bảng đếm tần suất** giữa 2 cột.

Tương đương `pivot_table` với `aggfunc='count'`, nhưng cú pháp gọn hơn.

```python
pd.crosstab(df['col_A'], df['col_B'])
pd.crosstab(df['col_A'], df['col_B'], normalize='index')  # tỷ lệ %
```

In [44]:
# Crosstab: số giao dịch theo merchant_category và is_fraud
ct = pd.crosstab(
    transactions['merchant_category'], 
    transactions['is_fraud']
)
print("Bảng chéo tần suất:")
display(ct)

# Tỷ lệ phần trăm theo hàng
ct_pct = pd.crosstab(
    transactions['merchant_category'],
    transactions['is_fraud'],
    normalize='index'
) * 100
print("\nTỷ lệ % fraud theo merchant_category:")
display(ct_pct.round(2))

Bảng chéo tần suất:


is_fraud,0,1
merchant_category,,
electronics,35163,1678
entertainment,35600,1757
food,35075,1644
groceries,35246,1721
shopping,35167,1662
travel,35115,1686



Tỷ lệ % fraud theo merchant_category:


is_fraud,0,1
merchant_category,,
electronics,95.45,4.55
entertainment,95.30,4.70
food,95.52,4.48
groceries,95.34,4.66
shopping,95.49,4.51
travel,95.42,4.58


## 5.9 Ứng dụng: Pivot Table trên dữ liệu đã merge

In [45]:
# Merge 3 bảng (nếu chưa có full_data)
full_data = pd.merge(users, accounts, on='user_id', how='inner')
full_data = pd.merge(full_data, transactions, on='account_id', how='inner')

# Pivot: amount trung bình theo risk_profile và merchant_category
pt_risk = pd.pivot_table(
    full_data,
    values='amount',
    index='risk_profile',
    columns='merchant_category',
    aggfunc='mean',
    margins=True
)
print("Amount trung bình theo risk_profile × merchant_category:")
display(pt_risk.round(2))

Amount trung bình theo risk_profile × merchant_category:


merchant_category,electronics,entertainment,food,groceries,shopping,travel,All
risk_profile,,,,,,,
high,2702.10,2728.62,2667.64,2763.93,2638.28,2637.41,2689.86
low,10718.03,10587.81,10531.31,10363.48,10962.15,11001.38,10693.33
medium,7228.20,7211.03,7174.37,7490.26,7389.20,7404.72,7315.83
All,4430.78,4447.07,4396.23,4470.44,4434.05,4462.27,4440.19


In [46]:
# Pivot: tỷ lệ fraud theo risk_profile và account_type
pt_fraud_risk = pd.pivot_table(
    full_data,
    values='is_fraud',
    index='risk_profile',
    columns='account_type',
    aggfunc='mean',
    margins=True
)
print("Tỷ lệ fraud theo risk_profile × account_type:")
display((pt_fraud_risk * 100).round(2))

Tỷ lệ fraud theo risk_profile × account_type:


account_type,current,savings,All
risk_profile,,,
high,4.59,4.52,4.56
low,4.65,4.57,4.61
medium,4.66,4.64,4.65
All,4.61,4.55,4.58


## Bài tập — Pivot Table

In [ ]:
# @title Bài tập 1
# Tạo pivot table: tổng amount theo txn_location và merchant_category
# aggfunc='sum', fill_value=0


In [ ]:
# @title Bài tập 2
# Tạo pivot table: số giao dịch theo txn_location và is_fraud
# aggfunc='count', margins=True


In [ ]:
# @title Bài tập 3
# Dùng pd.crosstab():
# Tạo bảng chéo giữa risk_profile và account_type (từ bảng full_data)
# Tính tỷ lệ % theo hàng (normalize='index')


In [ ]:
# @title Bài tập 4
# Merge users + accounts + transactions (nếu chưa có full_data)
# Tạo pivot table: amount trung bình theo (home_location) và (merchant_category)
# Thêm margins=True
# Merchant nào có amount cao nhất ở mỗi thành phố?


In [ ]:
# @title Bài tập 5 (Nâng cao)
# Tạo pivot table 2 cấp index:
# index=['risk_profile', 'account_type'], columns='is_fraud', values='amount'
# aggfunc=['mean', 'count']
# Nhóm nào (risk_profile + account_type) có amount fraud trung bình cao nhất?


# Quiz

In [49]:
# ============================
# QUIZ DATA (50 QUESTIONS)
# ============================

quiz_bank = [
    # === MERGE CƠ BẢN ===
    {"question": "pd.merge() dùng để làm gì?", "options": ["A) Kết hợp 2 DataFrame theo cột chung", "B) Xóa cột trong DataFrame", "C) Sắp xếp DataFrame"], "answer": "A"},
    {"question": "Kiểu merge mặc định (how) là gì?", "options": ["A) left", "B) inner", "C) outer"], "answer": "B"},
    {"question": "Inner merge giữ lại dòng nào?", "options": ["A) Chỉ dòng có key ở cả 2 bảng", "B) Tất cả dòng bảng trái", "C) Tất cả dòng cả 2 bảng"], "answer": "A"},
    {"question": "Left merge giữ lại dòng nào?", "options": ["A) Chỉ dòng có key ở cả 2 bảng", "B) Tất cả dòng bảng trái", "C) Tất cả dòng bảng phải"], "answer": "B"},
    {"question": "Right merge giữ lại dòng nào?", "options": ["A) Tất cả dòng bảng phải", "B) Tất cả dòng bảng trái", "C) Chỉ dòng trùng key"], "answer": "A"},
    {"question": "Outer merge giữ lại dòng nào?", "options": ["A) Chỉ dòng trùng key", "B) Tất cả dòng bảng trái", "C) Tất cả dòng từ cả 2 bảng"], "answer": "C"},
    {"question": "Tham số on trong merge dùng để?", "options": ["A) Chỉ định cột chung để ghép", "B) Chỉ định kiểu merge", "C) Chỉ định số dòng"], "answer": "A"},
    {"question": "Khi 2 bảng có tên cột key khác nhau, dùng tham số nào?", "options": ["A) on", "B) left_on và right_on", "C) key"], "answer": "B"},
    {"question": "Left merge điền gì cho dòng không khớp bên phải?", "options": ["A) 0", "B) NaN", "C) None"], "answer": "B"},
    {"question": "Inner merge tương đương lệnh SQL nào?", "options": ["A) INNER JOIN", "B) LEFT JOIN", "C) FULL OUTER JOIN"], "answer": "A"},
    {"question": "Kiểu merge nào cho kết quả nhiều dòng nhất?", "options": ["A) inner", "B) left", "C) outer"], "answer": "C"},
    {"question": "Kiểu merge nào cho kết quả ít dòng nhất?", "options": ["A) inner", "B) left", "C) outer"], "answer": "A"},

    # === MERGE NÂNG CAO ===
    {"question": "Khi 2 bảng có cột trùng tên (không phải key), merge tự thêm gì?", "options": ["A) Hậu tố _x, _y", "B) Tiền tố left_, right_", "C) Xóa cột trùng"], "answer": "A"},
    {"question": "Tham số suffixes dùng để?", "options": ["A) Tùy chỉnh hậu tố cho cột trùng tên", "B) Thêm tiền tố cho tất cả cột", "C) Xóa cột trùng tên"], "answer": "A"},
    {"question": "indicator=True tạo thêm cột nào?", "options": ["A) _merge", "B) _indicator", "C) _source"], "answer": "A"},
    {"question": "_merge='left_only' nghĩa là?", "options": ["A) Dòng chỉ có ở bảng trái", "B) Dòng chỉ có ở bảng phải", "C) Dòng có ở cả 2 bảng"], "answer": "A"},
    {"question": "_merge='both' nghĩa là?", "options": ["A) Dòng chỉ có ở bảng trái", "B) Dòng có ở cả 2 bảng", "C) Dòng chỉ có ở bảng phải"], "answer": "B"},
    {"question": "Để merge 3 bảng A, B, C ta cần?", "options": ["A) Merge A+B trước, rồi kết quả merge với C", "B) Merge cả 3 cùng lúc", "C) Không thể merge 3 bảng"], "answer": "A"},
    {"question": "validate='one_to_many' kiểm tra gì?", "options": ["A) Key bên trái là duy nhất", "B) Key bên phải là duy nhất", "C) Cả 2 key đều duy nhất"], "answer": "A"},
    {"question": "validate='one_to_one' báo lỗi khi nào?", "options": ["A) Khi key lặp ở bất kỳ bảng nào", "B) Khi không có key chung", "C) Khi kết quả có NaN"], "answer": "A"},
    {"question": "Merge tạo DataFrame mới hay thay đổi DataFrame gốc?", "options": ["A) Tạo DataFrame mới", "B) Thay đổi DataFrame gốc", "C) Tùy tham số inplace"], "answer": "A"},
    {"question": "Indicator hữu ích nhất khi muốn?", "options": ["A) Tìm dữ liệu không khớp giữa 2 bảng", "B) Tính thống kê", "C) Sắp xếp dữ liệu"], "answer": "A"},

    # === PIVOT ===
    {"question": "pivot() dùng để?", "options": ["A) Chuyển dữ liệu từ dạng dài sang rộng", "B) Ghép 2 bảng dữ liệu", "C) Lọc dữ liệu"], "answer": "A"},
    {"question": "Tham số index trong pivot() là?", "options": ["A) Cột làm hàng", "B) Cột làm cột mới", "C) Cột chứa giá trị"], "answer": "A"},
    {"question": "Tham số columns trong pivot() là?", "options": ["A) Cột làm hàng", "B) Cột làm tên cột mới", "C) Cột chứa giá trị"], "answer": "B"},
    {"question": "Tham số values trong pivot() là?", "options": ["A) Cột làm hàng", "B) Cột làm cột mới", "C) Cột chứa giá trị điền vào bảng"], "answer": "C"},
    {"question": "pivot() báo lỗi khi nào?", "options": ["A) Khi có giá trị trùng lặp", "B) Khi có NaN", "C) Khi có cột số"], "answer": "A"},
    {"question": "Khi pivot() bị lỗi trùng lặp, nên dùng gì thay thế?", "options": ["A) pivot_table()", "B) merge()", "C) groupby()"], "answer": "A"},
    {"question": "Dạng 'long format' là dữ liệu có đặc điểm gì?", "options": ["A) Nhiều dòng, ít cột", "B) Ít dòng, nhiều cột", "C) Dòng và cột bằng nhau"], "answer": "A"},
    {"question": "Dạng 'wide format' là dữ liệu có đặc điểm gì?", "options": ["A) Nhiều dòng, ít cột", "B) Ít dòng, nhiều cột", "C) Dòng và cột bằng nhau"], "answer": "B"},

    # === PIVOT TABLE ===
    {"question": "pivot_table() khác pivot() ở điểm nào?", "options": ["A) Có hàm tổng hợp (aggfunc)", "B) Chạy nhanh hơn", "C) Không cần tham số"], "answer": "A"},
    {"question": "aggfunc mặc định của pivot_table() là gì?", "options": ["A) sum", "B) mean", "C) count"], "answer": "B"},
    {"question": "aggfunc='sum' trong pivot_table() nghĩa là?", "options": ["A) Tính tổng giá trị", "B) Tính trung bình", "C) Đếm số lượng"], "answer": "A"},
    {"question": "aggfunc='count' trong pivot_table() nghĩa là?", "options": ["A) Tính tổng", "B) Tính trung bình", "C) Đếm số lượng"], "answer": "C"},
    {"question": "margins=True trong pivot_table() tạo thêm gì?", "options": ["A) Hàng và cột tổng cộng (All)", "B) Viền cho bảng", "C) Cột index"], "answer": "A"},
    {"question": "fill_value trong pivot_table() dùng để?", "options": ["A) Thay NaN bằng giá trị chỉ định", "B) Thêm dòng mới", "C) Xóa dòng NaN"], "answer": "A"},
    {"question": "Có thể dùng nhiều aggfunc cùng lúc không?", "options": ["A) Có, truyền list: aggfunc=['mean','sum']", "B) Không, chỉ được 1", "C) Có, nhưng phải gọi 2 lần"], "answer": "A"},
    {"question": "pivot_table với index là list sẽ tạo?", "options": ["A) Multi-level index (nhiều cấp)", "B) Lỗi", "C) Chỉ dùng phần tử đầu tiên"], "answer": "A"},
    {"question": "pd.crosstab() dùng để?", "options": ["A) Tạo bảng chéo tần suất", "B) Ghép bảng", "C) Chuyển đổi dạng dữ liệu"], "answer": "A"},
    {"question": "pd.crosstab() tương đương pivot_table với aggfunc nào?", "options": ["A) mean", "B) sum", "C) count/size"], "answer": "C"},
    {"question": "normalize='index' trong crosstab() nghĩa là?", "options": ["A) Tỷ lệ % theo hàng", "B) Tỷ lệ % theo cột", "C) Tỷ lệ % tổng thể"], "answer": "A"},
    {"question": "normalize='columns' trong crosstab() nghĩa là?", "options": ["A) Tỷ lệ % theo hàng", "B) Tỷ lệ % theo cột", "C) Tỷ lệ % tổng thể"], "answer": "B"},
    {"question": "normalize='all' trong crosstab() nghĩa là?", "options": ["A) Tỷ lệ % theo hàng", "B) Tỷ lệ % theo cột", "C) Tỷ lệ % trên toàn bảng"], "answer": "C"},

    # === TỔNG HỢP ===
    {"question": "Merge phù hợp nhất khi nào?", "options": ["A) Kết hợp dữ liệu từ nhiều bảng", "B) Chuyển đổi dạng dữ liệu", "C) Tính thống kê"], "answer": "A"},
    {"question": "Pivot phù hợp nhất khi nào?", "options": ["A) Kết hợp 2 bảng", "B) Chuyển dạng dài sang rộng", "C) Xóa dữ liệu trùng"], "answer": "B"},
    {"question": "Pivot table phù hợp nhất khi nào?", "options": ["A) Chuyển dạng + tổng hợp dữ liệu", "B) Chỉ chuyển dạng", "C) Chỉ tính trung bình"], "answer": "A"},
    {"question": "Để phân tích dữ liệu từ 3 bảng users-accounts-transactions, bước đầu tiên là?", "options": ["A) Merge các bảng lại", "B) Pivot dữ liệu", "C) Tính thống kê từng bảng"], "answer": "A"},
    {"question": "Kết quả merge có thể dùng làm input cho pivot_table không?", "options": ["A) Có", "B) Không", "C) Chỉ khi dùng inner merge"], "answer": "A"},
    {"question": "Crosstab thường dùng cho loại dữ liệu nào?", "options": ["A) Dữ liệu phân loại (categorical)", "B) Dữ liệu số liên tục", "C) Dữ liệu thời gian"], "answer": "A"},
]

In [50]:
# ============================
# QUIZ UI
# ============================

GREEN = "\033[92m"
RED = "\033[91m"
CYAN = "\033[96m"
YELLOW = "\033[93m"
MAGENTA = "\033[95m"
RESET = "\033[0m"

def show_menu():
    print(MAGENTA + "╔══════════════════════════════════╗" + RESET)
    print(MAGENTA + "║" + RESET + CYAN + "        🎯 QUIZ MENU ⭐           " + RESET + MAGENTA + "║" + RESET)
    print(MAGENTA + "╚══════════════════════════════════╝" + RESET)
    print(YELLOW + "1) 🔟 Random 10 câu" + RESET)
    print(YELLOW + "2) 🧩 Random 20 câu" + RESET)
    print(YELLOW + "3) 🔁 Luyện tập (hỏi đến khi đúng)" + RESET)
    print(YELLOW + "4) 📚 Thi toàn bộ 50 câu" + RESET)
    print(YELLOW + "5) 🚪 Thoát" + RESET)
    return input(CYAN + "👉 Chọn chế độ (1-5): " + RESET).strip()

def show_question_box(question, options):
    print(MAGENTA + "\n╔══════════════════════════════════╗" + RESET)
    print(MAGENTA + "║ " + RESET + CYAN + f"🧠 {question}" + RESET)
    print(MAGENTA + "╚══════════════════════════════════╝" + RESET)
    for opt in options:
        print("   " + opt)

def show_result(score, total):
    print(GREEN + "\n╔══════════════════════════════╗" + RESET)
    print(GREEN + f"  🎉 Kết quả: {score}/{total} 📊" + RESET)
    print(GREEN + "╚══════════════════════════════╝" + RESET)

In [51]:
# ============================
# QUIZ LOGIC
# ============================
import random

def shuffle_options(question_obj):
    options = question_obj["options"].copy()
    correct_answer = question_obj["answer"]
    correct_text = next(opt for opt in options if opt.startswith(correct_answer + ")"))
    random.shuffle(options)
    new_index = options.index(correct_text)
    question_obj["answer"] = ["A", "B", "C"][new_index]
    new_options = []
    for i, opt in enumerate(options):
        label = ["A", "B", "C"][i]
        text = opt.split(") ", 1)[1]
        new_options.append(f"{label}) {text}")
    question_obj["options"] = new_options
    return question_obj

def run_quiz(questions):
    score = 0
    for q in questions:
        q = shuffle_options(q.copy())
        show_question_box(q["question"], q["options"])
        ans = input("Trả lời (A/B/C): ").upper().strip()
        if ans == q["answer"]:
            print(GREEN + "✅ Chính xác!" + RESET)
            score += 1
        else:
            print(RED + "❌ Sai rồi!" + RESET)
            correct_opt = next(opt for opt in q["options"] if opt.startswith(q["answer"]))
            print(GREEN + f"👉 Đáp án đúng là: {correct_opt}" + RESET)
    show_result(score, len(questions))

def practice_mode(quiz_bank):
    print(CYAN + "\n=== 🔁 CHẾ ĐỘ LUYỆN TẬP ===" + RESET)
    for q in quiz_bank:
        q = shuffle_options(q.copy())
        while True:
            show_question_box(q["question"], q["options"])
            ans = input("Trả lời (A/B/C): ").upper().strip()
            if ans == q["answer"]:
                print(GREEN + "✅ Chính xác!" + RESET)
                break
            else:
                print(RED + "❌ Sai rồi, thử lại nhé." + RESET)

while True:
    choice = show_menu()
    if choice == "1":
        run_quiz(random.sample(quiz_bank, 10))
    elif choice == "2":
        run_quiz(random.sample(quiz_bank, 20))
    elif choice == "3":
        practice_mode(quiz_bank)
    elif choice == "4":
        run_quiz(quiz_bank)
    elif choice == "5":
        print(GREEN + "👋 Tạm biệt, chúc bạn học tốt!" + RESET)
        break
    else:
        print(RED + "⚠️ Lựa chọn không hợp lệ, thử lại nhé." + RESET)

╔══════════════════════════════════╗
║        🎯 QUIZ MENU ⭐           ║
╚══════════════════════════════════╝
1) 🔟 Random 10 câu
2) 🧩 Random 20 câu
3) 🔁 Luyện tập (hỏi đến khi đúng)
4) 📚 Thi toàn bộ 50 câu
5) 🚪 Thoát

╔══════════════════════════════════╗
║ 🧠 Để merge 3 bảng A, B, C ta cần?
╚══════════════════════════════════╝
   A) Merge A+B trước, rồi kết quả merge với C
   B) Merge cả 3 cùng lúc
   C) Không thể merge 3 bảng
✅ Chính xác!

╔══════════════════════════════════╗
║ 🧠 Khi pivot() bị lỗi trùng lặp, nên dùng gì thay thế?
╚══════════════════════════════════╝
   A) pivot_table()
   B) groupby()
   C) merge()
✅ Chính xác!

╔══════════════════════════════════╗
║ 🧠 _merge='left_only' nghĩa là?
╚══════════════════════════════════╝
   A) Dòng chỉ có ở bảng phải
   B) Dòng có ở cả 2 bảng
   C) Dòng chỉ có ở bảng trái
❌ Sai rồi!
👉 Đáp án đúng là: C) Dòng chỉ có ở bảng trái

╔══════════════════════════════════╗
║ 🧠 Tham số columns trong pivot() là?
╚══════════════════════════════════╝
 